In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.04', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44
1,IIT: Pct Change due to behavior,-2.58,-2.08,-1.36,-0.74,-0.11,0.53,1.29,2.21,3.42,5.05,0.56,8.27
2,IIT: Pct Change due to macro,5.25,10.64,16.41,22.58,29.22,36.32,43.91,51.98,60.54,69.61,35.05,62.88
3,IIT: Overall Pct Change in taxes,-3.05,2.44,8.58,15.06,22.06,29.59,37.83,46.88,57.00,68.49,28.41,66.75
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,15.83,34.94,56.29,79.37,104.42,131.48,160.86,192.82,227.93,266.96,127.05,255.13
6,CIT: Pct Change due to macro,-11.11,-18.32,-24.41,-29.53,-33.90,-37.63,-40.86,-43.67,-46.12,-48.26,-37.33,-50.04
7,CIT: Overall Pct Change in taxes,2.96,10.21,18.13,26.39,35.13,44.37,54.26,64.95,76.70,89.85,42.28,77.42
8,All: Pct Change due to tax rates,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.12
9,All: Pct Change due to behavior,-1.47,0.16,2.13,4.11,6.23,8.48,10.99,13.80,17.08,21.00,8.23,23.51


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.26,-0.28,-0.30,-0.30,-0.31,-0.33,-0.34,-0.35,-0.37,-0.38,-3.22
9,Rev Change Due to Behavior,-0.07,0.01,0.12,0.24,0.38,0.54,0.73,0.95,1.23,1.57,5.70
10,Rev Change Due to Macro,0.21,0.45,0.73,1.02,1.34,1.70,2.12,2.55,3.04,3.57,16.72
11,Total Revenue Change,-0.14,0.16,0.53,0.93,1.40,1.93,2.58,3.31,4.17,5.20,20.07


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.23,-0.25,-0.27,-0.28,-0.29,-0.30,-0.31,-0.33,-0.34,-0.35,-2.97
1,Rev Change Due to Behavior,-0.11,-0.10,-0.07,-0.04,-0.01,0.03,0.07,0.13,0.21,0.33,0.46
2,Rev Change Due to Macro,0.22,0.50,0.82,1.17,1.57,2.02,2.54,3.12,3.77,4.51,20.24
3,Total Revenue Change,-0.13,0.11,0.43,0.78,1.18,1.65,2.19,2.81,3.55,4.44,17.01


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.22,-0.25,-0.26,-0.27,-0.28,-0.29,-0.30,-0.32,-0.33,-0.34,-2.86
1,Rev Change Due to Behavior,-0.11,-0.09,-0.06,-0.04,-0.01,0.03,0.07,0.13,0.21,0.32,0.45
2,Rev Change Due to Macro,0.21,0.49,0.78,1.12,1.50,1.95,2.45,3.02,3.66,4.38,19.56
3,Total Revenue Change,-0.12,0.11,0.41,0.74,1.14,1.59,2.11,2.73,3.45,4.31,16.45


In [8]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95
1,Rev Change Due to Behavior,-0.11,-0.09,-0.06,-0.03,-0.01,0.03,0.07,0.12,0.19,0.30,0.42
2,Rev Change Due to Macro,-0.01,-0.01,-0.01,-0.01,-0.00,0.01,0.03,0.06,0.12,0.21,0.39
3,Total Revenue Change,-0.11,-0.39,-0.37,-0.35,-0.32,-0.29,-0.24,-0.16,-0.04,0.14,-2.14


In [10]:
result_df_static

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.08,4.56,4.75,4.94,5.15,5.36,5.59,5.81,6.05,6.29,52.57
Reform,4.08,4.27,4.45,4.63,4.83,5.04,5.25,5.47,5.69,5.92,49.63
Difference,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95


In [9]:
df_levels.to_csv('og_usa_result_w_tcja_prod_4.csv')